# MobileNetV2 on ADNI MRI — trained FROM SCRATCH

**Update to this notebook.** It previously fine-tuned MobileNetV2 from ImageNet weights.
That approach was tested three ways on this dataset and every variant lost to a much
smaller from-scratch CNN, so this notebook now trains the same architecture from random
initialization as the primary approach. The pretrained results are kept at the bottom as
a documented negative result, because "we tried it and it hurt" is a finding worth
keeping, not a mistake worth hiding.

| approach | subject-level accuracy |
|---|---|
| MobileNetV2, ImageNet weights, full unfreeze | 34.8% |
| MobileNetV2, ImageNet weights, partial unfreeze + frozen BN | ~37% |
| MobileNetV2, ImageNet weights, partial unfreeze + adaptive BN | 40.9% |
| **MobileNetV2, from scratch (this notebook)** | **see below** |
| custom SimpleCNN, from scratch (notebook 02) | 56.1% |

**Why pretraining hurts here.** ImageNet weights encode statistics of natural colour
photographs. Our input is a grayscale MRI slice copied into three identical channels —
no colour information, completely different spatial statistics, and a foreground that
fills the frame. The domain gap is large enough that the pretrained features are a worse
starting point than random ones, and with only ~300 training subjects there isn't enough
data to retrain them out. This is "negative transfer", and it is the expected outcome
when the source and target domains are this far apart.

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from datasets import CLASSES, build_dataloaders, build_dataloaders_25d, compute_class_weights
from models import SimpleCNN, build_mobilenetv2, build_efficientnet_b0
from train import train_model
from evaluate import (get_predictions, slice_level_report, subject_level_report,
                      subject_level_soft_vote, ensemble_predictions)

%matplotlib inline
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '-', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU only')

manifest = pd.read_csv('../data/manifest.csv')
class_weights = compute_class_weights(manifest)
torch.manual_seed(42); np.random.seed(42)
print(manifest.groupby(['class', 'split']).size().unstack())

In [ ]:
def plot_cm(cm, title, ax=None):
    """Confusion matrix with per-class recall on the diagonal made obvious --
    aggregate accuracy hides which class the model is actually failing on."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title(title)
    return ax

## 1. Data loaders

`rgb=True` gives the 3-channel ImageNet-normalized transforms. The normalization
constants are arbitrary for a from-scratch model, but they are kept identical to the
pretrained runs so that **weight initialization is the only variable that changed**
between this result and the negative result at the bottom.

In [ ]:
train_loader, val_loader, test_loader = build_dataloaders(
    manifest, batch_size=32, num_workers=2, rgb=True)
print('class weights:', dict(zip(CLASSES, [round(w, 3) for w in class_weights.tolist()])))

## 2. Model — random initialization

`pretrained=False` is the whole point of this notebook.

In [ ]:
model = build_mobilenetv2(num_classes=len(CLASSES), pretrained=False)
n_params = sum(p.numel() for p in model.parameters())
print(f'MobileNetV2 from scratch: {n_params:,} parameters')

## 3. Train

Single phase, not the two-phase freeze/unfreeze schedule the pretrained version used —
there are no pretrained features to protect, so there is nothing to freeze. Learning rate
is 1e-3 rather than the 1e-5 used for fine-tuning, because random weights need to move a
long way. Early stopping restores the best-validation-loss checkpoint.

In [ ]:
history = train_model(
    model, train_loader, val_loader, class_weights, device,
    epochs=40, lr=1e-3, patience=7, weight_decay=1e-4,
    checkpoint_path='../models/checkpoints/mobilenetv2_honest2d.pt')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history['train_loss'], label='train'); axes[0].plot(history['val_loss'], label='val')
axes[0].set_title('loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
axes[1].plot(history['train_acc'], label='train'); axes[1].plot(history['val_acc'], label='val')
axes[1].set_title('accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()
plt.tight_layout()

## 4. Evaluate

Three numbers, and the gap between them matters:

- **slice level** — every axial slice judged independently. Not what a clinician does.
- **subject level, hard vote** — majority across a subject's 32 slices.
- **subject level, soft vote** — average the probability distributions instead. Keeps
  confidence information that majority voting throws away.

In [ ]:
preds = get_predictions(model, test_loader, device)

print('===== SLICE LEVEL =====')
cm_slice = slice_level_report(preds)

print('\n===== SUBJECT LEVEL — hard majority vote =====')
cm_hard, subj_hard = subject_level_report(preds)

print('\n===== SUBJECT LEVEL — soft vote =====')
cm_soft, subj_soft = subject_level_soft_vote(preds)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_cm(cm_slice, 'slice level', axes[0])
plot_cm(cm_hard, 'subject, hard vote', axes[1])
plot_cm(cm_soft, 'subject, soft vote', axes[2])
plt.tight_layout()

### Sample predictions

Per-subject output with the averaged class probabilities, so the failures are inspectable
rather than hidden behind an accuracy number.

In [ ]:
subj_soft['correct'] = subj_soft['true'] == subj_soft['pred']
print('correct:', int(subj_soft['correct'].sum()), 'of', len(subj_soft))
display(subj_soft.head(15).round(3))
print('\nmisclassified subjects:')
display(subj_soft[~subj_soft['correct']].round(3))

## 5. Negative result kept on the record — ImageNet fine-tuning

Not re-run here (it takes GPU time to reproduce a worse answer), but the numbers are
preserved in `reports/metrics_legacy_pretrained.json`. Two lessons worth keeping:

1. **Don't freeze BatchNorm** when fine-tuning onto a distant domain. Freezing it was
   tried on the theory that it would reduce overfitting; it made things worse, because it
   forces the network to keep normalizing with ImageNet-photo statistics instead of
   adapting to the actual MRI distribution.
2. **Pretrained is not automatically better.** It is better when the source and target
   domains are close. Grayscale medical volumes and natural photographs are not close.

In [ ]:
legacy = json.load(open('../reports/metrics_legacy_pretrained.json'))
print(json.dumps(legacy.get('mobilenetv2', {}), indent=2))